In [ ]:
import pandas as pd

In [ ]:
import matplotlib.pyplot as plt

### Datasets
We have 5 csv file 

   1. dim_date.csv  
   2. dim_hotels.csv
   3. dim_rooms.csv
   4. fact_aggregated_bookings
   5. fact_bookings.csv

 ### **Read bookings data in a datagrame**

In [ ]:
df_bookings = pd.read_csv('datasets for Hospitality project/fact_bookings.csv')
df_bookings.head()

**Explore bookings data**

In [ ]:
df_bookings.room_category.unique()

In [ ]:
df_bookings.booking_platform.value_counts()

### Bar graph on method of booking

In [ ]:
df_bookings.booking_platform.value_counts().plot(
    kind="barh",
    figsize=(6, 4.5),
    color=["khaki", "skyblue", "lightgreen", "lavender", "lightcoral", "orange", "plum"]
)

plt.title("Bar graph on method of booking", fontsize=13)
plt.xlabel("Number of Bookings")
plt.ylabel("Booking Platform")
plt.tight_layout()
plt.show()

### ***Loading rest of the data files***

In [ ]:
df_date = pd.read_csv('datasets for Hospitality project/dim_date.csv')
df_hotels = pd.read_csv('datasets for Hospitality project/dim_hotels.csv')
df_rooms = pd.read_csv('datasets for Hospitality project/dim_rooms.csv')
df_agg_bookings = pd.read_csv('datasets for Hospitality project/fact_aggregated_bookings.csv')

### Exploring Hotels

In [ ]:
df_hotels.head(5)

In [ ]:
df_hotels.category.value_counts()

In [ ]:
 df_hotels.city.value_counts().sort_values()

### Bar chart on top cities where hotel chains are situated

In [ ]:
df_hotels.city.value_counts().sort_values().plot(kind="bar", figsize=(5,5), color=["khaki","Skyblue","lightgreen","lavender"])

###  ***Explore aggregate bookings***



In [ ]:
df_agg_bookings.head(3)

###  Find out unique property ids in aggregate bookings dataset


In [ ]:
df_agg_bookings.property_id.unique()

###  Find out total bookings per property_id


In [ ]:
df_agg_bookings.groupby("property_id")["successful_bookings"].sum()

### Find out days on which bookings are greater than capacity

In [ ]:
df_agg_bookings[df_agg_bookings.successful_bookings>df_agg_bookings.capacity]

### Find out top 5 properties that have highest capacity

In [ ]:
df_agg_bookings.sort_values(by='capacity', ascending=False).head(5)


### ***Data Cleaning***

In [ ]:
df_bookings.describe()

**(1) Clean invalid guests**

In [ ]:
df_bookings[df_bookings.no_guests<=0]

### After cleaning all of invalid number of guest records

In [ ]:
df_bookings[df_bookings.no_guests>=0]

### Outlier removal in revenue generated by implementingstandard deviation

In [ ]:
df_bookings.revenue_generated.min(), df_bookings.revenue_generated.max()

In [ ]:
avg, std =df_bookings.revenue_generated.mean(),df_bookings.revenue_generated.std()
avg, std

In [ ]:
higher_limit = avg + 3*std
higher_limit

In [ ]:
lower_limit = avg - 3*std
lower_limit

### Final outcome after data cleaning

In [ ]:
df_bookings = df_bookings[df_bookings.revenue_generated<=higher_limit]
df_bookings

In [ ]:
df_bookings.shape

### **Filling na values in aggregate bookings**

In [ ]:
df_agg_bookings.isnull().sum()

In [ ]:
df_agg_bookings[df_agg_bookings.capacity.isna()]

In [ ]:
df_agg_bookings.capacity.median()

In [ ]:
df_agg_bookings["capacity"] = df_agg_bookings["capacity"].fillna(df_agg_bookings["capacity"].median())
df_agg_bookings.loc[[8,15]]

### **In aggregate bookings filtering records that have successful_bookings value greater than capacity**

In [ ]:
df_agg_bookings[df_agg_bookings.successful_bookings>df_agg_bookings.capacity]


In [ ]:
df_agg_bookings.shape

In [ ]:
df_agg_bookings = df_agg_bookings[df_agg_bookings.successful_bookings<=df_agg_bookings.capacity]
df_agg_bookings.shape

In [ ]:
df_agg_bookings[df_agg_bookings.successful_bookings>df_agg_bookings.capacity]

### Data Transformation

In [ ]:
df_agg_bookings.head()

In [ ]:
df_agg_bookings["Occupancy %"] = df_agg_bookings["successful_bookings"]/df_agg_bookings["capacity"]

In [ ]:
df_agg_bookings["Occupancy %"] = df_agg_bookings["Occupancy %"].apply( lambda X : round (X*100, 2 ))

In [ ]:
df_agg_bookings.head()

### Insights Generation

**1. What is an average occupancy rate in each of the room categories?**

In [ ]:
df = pd.merge(df_agg_bookings,df_rooms, left_on = "room_category", right_on = "room_id")
df.head(3)

In [ ]:
df.drop("room_id", axis =1 , inplace = True)
df.head(3)

In [ ]:
df.groupby("room_class")["Occupancy %"].mean()

**2. Average occupancy rate per city**

In [ ]:
df_hotels.head(3)

In [ ]:
df = pd.merge(df,df_hotels, on = "property_id")
df.head(3)

In [ ]:
df.groupby("city")["Occupancy %"].mean().plot(kind="bar", figsize=(5,5), color=["khaki","Skyblue","lightgreen","lavender"])

plt.title("Average Occupancy Percentage by City")
plt.xlabel("City")
plt.ylabel("Average Occupancy (%)")
plt.tight_layout()

plt.show()


**3. When was the occupancy better? Weekday or Weekend?**

In [ ]:
df = pd.merge(df, df_date, left_on="check_in_date", right_on="date")
df.head(3)

In [ ]:
df.groupby("day_type")["Occupancy %"].mean().round(2)

In [ ]:
df.groupby("day_type")["Occupancy %"].mean().round(2).plot(kind="bar", figsize=(4,5), color=["khaki","lightgreen"])

plt.title("Average Occupancy Percentage by weeks")
plt.ylabel("Average Occupancy (%)")
plt.xlabel("Day Type")
plt.tight_layout()
plt.show()

**4. In the month of July, what is the occupancy for different cities**

In [ ]:
df["mmm yy"].unique()

In [ ]:
df_july = df[df["mmm yy"]=="Jul 22"]
df_july.head(4)

In [ ]:
df_july.groupby("city")["Occupancy %"].mean().round(2).sort_values(ascending=False)

**5: We got new data for the month of august. Append that to existing data**

In [ ]:
df_august = pd.read_csv("datasets for Hospitality project/new_data_august.csv")
df_august.head(5)

**Joining august with existing df row wise to make a new Data frame**

In [ ]:
latest_df = pd.concat([df,df_august], ignore_index = True , axis=0)
latest_df.tail(5)

In [ ]:
df_hotels.head(5)

**6. Print revenue realized per city**

In [ ]:
df_bookings_all = pd.merge(df_bookings, df_hotels, on="property_id")
df_bookings_all.head(3)

In [ ]:
df_bookings_all.groupby("city")["revenue_realized"].sum().sort_values(ascending=False)

In [ ]:
revenue_by_city = (
    df_bookings_all.groupby("city")["revenue_realized"].sum().sort_values(ascending=False) / 1e6)

revenue_by_city.plot(
    kind="bar",
    figsize=(5, 5),
    color=["khaki", "lavender", "skyblue", "lightgreen"])

plt.title("Revenue Realized by City")
plt.xlabel("City")
plt.ylabel("Revenue Realized (in Millions)")
plt.tight_layout()
plt.show()

**7 Print month by month revenue**

In [ ]:
df_date.head(3)

In [ ]:
df_bookings_all.head(3)

In [ ]:
df_date.info()

**Changing data type of date formation into datetime64**

In [ ]:
df_date["date"] = pd.to_datetime(df_date["date"], format="%Y-%m-%d")
df_date.head(3)

In [ ]:
df_bookings_all["check_in_date"] = pd.to_datetime(df_bookings_all["check_in_date"], format='mixed')
df_bookings_all.head(4)

In [ ]:
df_bookings_all = pd.merge(df_bookings_all, df_date, left_on="check_in_date", right_on="date")
df_bookings_all.head(3)

In [ ]:
df_bookings_all.groupby("mmm yy")["revenue_realized"].sum()

**8. Print revenue realized per hotel type**

In [ ]:
df_bookings_all.property_name.unique()

In [ ]:
df_bookings_all.groupby("property_name")["revenue_realized"].sum().round(2).sort_values()